# A convolution layer and a patch embedding in NumPy

The chapter's network slides two 3 × 3 filters over a 6 × 6 image with one channel, one
pixel at a time. A layer in a real network takes several input channels and several
filters, and it has a stride and padding. A Vision Transformer's patch embedding is the
same layer with the stride equal to the filter size, computed as one matrix product. Here
you write both with NumPy, in PyTorch's layout: an image is channels × height × width, and
a stack of filters is output channels × input channels × $k$ × $k$.

**Working through it.** Each part asks for code in the cell marked `# TODO`. Replace the
`raise NotImplementedError` with your code and run the check cell below it. The check prints
`Part N passed`, or stops with a message about what is wrong.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# The running example: one channel of 6 x 6 pixels, a bright line in column 3.
IMAGE = np.zeros((1, 6, 6))
IMAGE[0, :, 2] = 1.0

# Its two filters, stacked as 2 output channels x 1 input channel x 3 x 3.
VERTICAL = np.array([[-1, 2, -1], [-1, 2, -1], [-1, 2, -1]], dtype=float)
FILTERS = np.stack([VERTICAL, VERTICAL.T])[:, None]
print("image", IMAGE.shape, "filters", FILTERS.shape)

## Part 1 (coding task): A convolution layer

Write `conv2d(x, w, stride=1, padding=0)` for an image `x` of shape
$(C_\text{in}, H, W)$ and filters `w` of shape $(C_\text{out}, C_\text{in}, k, k)$.

- First surround each channel with `padding` rows and columns of zeros, on all four sides.
- Then place each filter at every position, moving it `stride` pixels at a time, and add
  up all $C_\text{in} \cdot k \cdot k$ products of the filter with the window under it.

With $s$ the stride, that is

$$
\text{out}[o, i, j] = \sum_{c} \sum_{a} \sum_{b} \; w[o, c, a, b] \; x_\text{pad}[c,\ i s + a,\ j s + b].
$$

The filter is not flipped: this is what PyTorch's `Conv2d` computes and what the chapter
calls a convolution.

The output has shape $(C_\text{out}, H_\text{out}, W_\text{out})$ with
$H_\text{out} = \lfloor (H + 2p - k) / s \rfloor + 1$, where $p$ is the padding, and
$W_\text{out}$ found the same way. Loops over the output positions are fine here.

In [ ]:
def conv2d(x, w, stride=1, padding=0):
    """x: (C_in, H, W) image, w: (C_out, C_in, k, k) filters. Returns (C_out, H_out, W_out)."""
    # TODO
    raise NotImplementedError

In [ ]:
maps = conv2d(IMAGE, FILTERS)
print(maps[0])
assert maps.shape == (2, 4, 4), f"the running example should give 2 maps of 4 x 4, got shape {maps.shape}"
assert np.allclose(maps[0], [[-3, 6, -3, 0]] * 4), "the vertical map should be -3, 6, -3, 0 in every row"
assert np.allclose(maps[1], 0), "the horizontal map should be 0 everywhere"

# Two input channels, an uneven filter and a 4 x 5 image.
x = np.array([[[1, 0, 2, 0, 1], [0, 3, 0, 1, 0], [2, 0, 1, 0, 2], [0, 1, 0, 3, 0]],
              [[0, 2, 0, 1, 1], [1, 0, 1, 0, 0], [0, 1, 2, 1, 0], [3, 0, 0, 0, 1]]], dtype=float)
w = np.array([[[[1, 2, 0], [0, 1, -1], [-2, 0, 1]],
               [[0, 1, 1], [-1, 0, 2], [1, 0, 0]]]], dtype=float)
out = conv2d(x, w)
assert out.shape == (1, 2, 3), f"a 4 x 5 image and a 3 x 3 filter give one map of 2 x 3, got shape {out.shape}"
assert not np.allclose(out, [[[3, 3, 9], [2, 6, 9]]]), "the filter is flipped: slide it as it is, the way Conv2d does"
assert not np.allclose(out, [[[1, 3, 3], [5, 5, 0]]]), "only the first input channel is counted: add the products over every input channel"
assert np.allclose(out, [[[4, 5, 6], [13, 7, -2]]]), f"stride 1 and no padding should give [[4, 5, 6], [13, 7, -2]], got {out[0].tolist()}"

out = conv2d(x, w, stride=2, padding=1)
assert out.shape == (1, 2, 3), f"with stride 2 and padding 1 the map should be 2 x 3, got shape {out.shape}"
assert np.allclose(out, [[[8, -3, -2], [6, 7, -4]]]), f"stride 2 and padding 1 should give [[8, -3, -2], [6, 7, -4]], got {out[0].tolist()}"

assert conv2d(IMAGE, FILTERS, padding=1).shape == (2, 6, 6), "padding 1 should keep the maps at 6 x 6"
size = conv2d(np.ones((1, 8, 8)), np.ones((1, 1, 3, 3)), stride=2).shape
assert size == (1, 3, 3), f"an 8 x 8 image, a 3 x 3 filter and stride 2 give a 3 x 3 map, got {size[1:]}"
print("Part 1 passed")

## Part 2 (coding task): Patches and a patch embedding

A patch embedding cuts the image into non-overlapping $p \times p$ patches, flattens each
one and multiplies it by a learned matrix. Part 1's layer with the stride equal to the
filter size computes the same numbers, one window per patch. Libraries use the matrix
form, which needs no loops.

Write `patches(x, p)` for `x` of shape $(C, H, W)$, with $H$ and $W$ multiples of $p$. It
returns a matrix with one row per patch and $C p^2$ columns.

- The rows go through the patches in reading order, along the top row of patches from left
  to right, then along the next row of patches.
- Each row lists the patch's values in the order a filter of shape $(C, p, p)$ flattens:
  all of channel 0 row by row, then all of channel 1, and so on.

Use `reshape` and `transpose`, with no Python loops.

Then write `patch_embed(x, w)` for filters `w` of shape $(D, C, p, p)$: one matrix product
of `patches(x, p)` with the flattened filters, arranged into shape $(D, H / p, W / p)$. It
must give the same numbers as `conv2d(x, w, stride=p)`.

In [ ]:
def patches(x, p):
    """x: (C, H, W). Returns (H // p * W // p, C * p * p), one row per patch in reading order."""
    # TODO
    raise NotImplementedError


def patch_embed(x, w):
    """x: (C, H, W), w: (D, C, p, p). Returns (D, H // p, W // p), using one matrix product."""
    # TODO
    raise NotImplementedError

In [ ]:
x = np.arange(2 * 4 * 6).reshape(2, 4, 6)
P = patches(x, 2)
print(P)
assert P.shape == (6, 8), f"a 2 x 4 x 6 image in 2 x 2 patches gives 6 rows of 8 values, got shape {P.shape}"
assert not np.array_equal(P[0], np.arange(8)), "a plain reshape mixes values from different patches: move the axes with transpose before flattening"
assert not np.array_equal(P[0], [0, 24, 1, 25, 6, 30, 7, 31]), "each row should list all of channel 0 first, then channel 1, as w[d].ravel() does"
assert np.array_equal(P[0], [0, 1, 6, 7, 24, 25, 30, 31]), f"row 0 should be the top-left patch, channel 0 row by row and then channel 1, got {P[0]}"
assert not np.array_equal(P[1], [12, 13, 18, 19, 36, 37, 42, 43]), "the rows should go along the top row of patches first, then down"
assert np.array_equal(P[1], [2, 3, 8, 9, 26, 27, 32, 33]), f"row 1 should be the second patch of the top row, got {P[1]}"
assert np.array_equal(P[3], [12, 13, 18, 19, 36, 37, 42, 43]), f"row 3 should be the first patch of the second row, got {P[3]}"

line = patch_embed(IMAGE, FILTERS[:1])
assert line.shape == (1, 2, 2), f"one filter on four 3 x 3 patches gives shape (1, 2, 2), got {line.shape}"
assert np.allclose(line, [[[-3, 0], [-3, 0]]]), f"the vertical filter on the running example's patches gives -3, 0 in each row, got {line[0].tolist()}"

three_channels = np.random.default_rng(7).normal(size=(3, 6, 9))
four_filters = np.random.default_rng(8).normal(size=(4, 3, 3, 3))
emb = patch_embed(three_channels, four_filters)
assert emb.shape == (4, 2, 3), f"4 filters on a 6 x 9 image in 3 x 3 patches give shape (4, 2, 3), got {emb.shape}"
assert np.allclose(emb, conv2d(three_channels, four_filters, stride=3)), "patch_embed should give the same numbers as conv2d with the stride equal to the patch size"
print("Part 2 passed")

## Solution

### Part 1: A convolution layer

In [ ]:
def conv2d(x, w, stride=1, padding=0):
    """x: (C_in, H, W) image, w: (C_out, C_in, k, k) filters. Returns (C_out, H_out, W_out)."""
    x = np.pad(x, ((0, 0), (padding, padding), (padding, padding)))   # not the channel axis
    c_out, c_in, k, _ = w.shape
    h_out = (x.shape[1] - k) // stride + 1
    w_out = (x.shape[2] - k) // stride + 1
    out = np.zeros((c_out, h_out, w_out))
    for i in range(h_out):
        for j in range(w_out):
            window = x[:, i * stride:i * stride + k, j * stride:j * stride + k]
            out[:, i, j] = (w * window).sum(axis=(1, 2, 3))           # every filter at once
    return out

**Why it works.** `np.pad` with a width for each axis pads only the height and the width. Called
with one number, it would pad the channel axis as well. Window $(i, j)$ starts at row $i s$ and
column $j s$ of the padded image, and the starts run from 0 in steps of $s$ up to the last start
that still fits the filter, which is why there are $\lfloor (H + 2p - k) / s \rfloor + 1$
positions. Multiplying the window, of shape $(C_\text{in}, k, k)$, by `w`, of shape
$(C_\text{out}, C_\text{in}, k, k)$, broadcasts it against every filter, and summing over the
last three axes adds up each filter's $C_\text{in} k^2$ products, the sum over input channels
included.

### Part 2: Patches and a patch embedding

In [ ]:
def patches(x, p):
    """x: (C, H, W). Returns (H // p * W // p, C * p * p), one row per patch in reading order."""
    c, h, w = x.shape
    grid = x.reshape(c, h // p, p, w // p, p)     # channel, patch row, row in patch, patch column, column in patch
    grid = grid.transpose(1, 3, 0, 2, 4)          # patch row, patch column, channel, row in patch, column in patch
    return grid.reshape((h // p) * (w // p), c * p * p)


def patch_embed(x, w):
    """x: (C, H, W), w: (D, C, p, p). Returns (D, H // p, W // p), using one matrix product."""
    d, c, p, _ = w.shape
    tokens = patches(x, p) @ w.reshape(d, -1).T   # one row per patch, one column per filter
    return tokens.T.reshape(d, x.shape[1] // p, x.shape[2] // p)

**Why it works.** `reshape(C, H/p, p, W/p, p)` only relabels the stored order: a row of
the image of length $W$ is split into $W/p$ pieces of $p$, and the $H$ rows into
$H/p$ groups of $p$. `transpose(1, 3, 0, 2, 4)` brings the two patch indices to the front
in reading order and leaves channel, row and column of each patch in the order
`w[d].ravel()` uses, so the final `reshape` gives each patch as one row lined up with each
flattened filter. The product has one row per patch and one column per filter, the token
vectors of a ViT. Transposed and reshaped, the same numbers form $D$ maps of
$H/p \times W/p$, which is what a convolution with stride $p$ returns. Without the last
transpose, `reshape` would pour the numbers of the first patches into the first map.